# 99 — Emission Factor Verification
**Purpose:** Cross-check all emission factors in `data/emission_factors.py` against 
their primary sources.

**Sources verified:**
- ADEME Base Carbone® v23.10 (downloaded 19/05/2026)
- Umweltbundesamt (UBA) 2024
- DESNZ 2024
- EEA 2024

**Author:** Emma McCallum  
**Date:** May 2026

In [1]:
!pip install duckdb --break-system-packages

In [2]:
!pip install pandas --break-system-packages

In [3]:
import pandas as pd

df = pd.read_csv('../references/Base_Carbone_V23.10.csv', 
                 sep=';', 
                 encoding='latin1', 
                 nrows=2)
print(df.columns.tolist())
df

['Type Ligne', "Identifiant de l'élément", 'Structure', "Type de l'élément", "Statut de l'élément", 'Nom base français', 'Nom base anglais', 'Nom base espagnol', 'Nom attribut français', 'Nom attribut anglais', 'Nom attribut espagnol', 'Nom frontière français', 'Nom frontière anglais', 'Nom frontière espagnol', 'Code de la catégorie', 'Tags français', 'Tags anglais', 'Tags espagnol', 'Unité français', 'Unité anglais', 'Unité espagnol', 'Contributeur', 'Autres Contributeurs', 'Programme', 'Url du programme', 'Source', 'Localisation géographique', 'Sous-localisation géographique français', 'Sous-localisation géographique anglais', 'Sous-localisation géographique espagnol', 'Date de création', 'Date de modification', 'Période de validité', 'Incertitude', 'Réglementations', 'Transparence', 'Qualité', 'Qualité TeR', 'Qualité GR', 'Qualité TiR', 'Qualité C', 'Qualité P', 'Qualité M', 'Commentaire français', 'Commentaire anglais', 'Commentaire espagnol', 'Type poste', 'Nom poste français', 'N

,Type Ligne,Identifiant de l'élément,Structure,Type de l'élément,Statut de l'élément,Nom base français,Nom base anglais,Nom base espagnol,Nom attribut français,Nom attribut anglais,...,Code gaz supplémentaire 2,Valeur gaz supplémentaire 2,Code gaz supplémentaire 3,Valeur gaz supplémentaire 3,Code gaz supplémentaire 4,Valeur gaz supplémentaire 4,Code gaz supplémentaire 5,Valeur gaz supplémentaire 5,Autres GES,CO2b
0,Elément,34052,élément non décomposé,Facteur d'émission,Archivé,"""""""\tSalade César au poulet (salade verte""""""",Caesar's salad (salad,NaN,"fromage, croûtos, sauce)","chicken, croûtons, sauce)",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Elément,33558,élément non décomposé,Facteur d'émission,Archivé,"""""""Brioche fourrée crème pâtissière (type """"""""...",Brioche,NaN,préemballée,"filled with custard (Chinese brioche type), pr...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
df_full = pd.read_csv('../references/Base_Carbone_V23.10.csv', 
                      sep=';', 
                      encoding='latin1')

# Search for natural gas entries
mask = df_full['Nom base français'].str.contains('gaz naturel', case=False, na=False)
df_full[mask][['Nom base français', 'Unité français', 'Total poste non décomposé', 'Statut de l\'élément']]

/tmp/ipykernel_1245/1681663031.py:1: DtypeWarning: Columns (0: Nom attribut espagnol, 1: Tags espagnol, 2: Sous-localisation géographique espagnol, 3: Transparence, 4: Code gaz supplémentaire 1, 5: Valeur gaz supplémentaire 1, 6: Code gaz supplémentaire 2, 7: Valeur gaz supplémentaire 2, 8: Code gaz supplémentaire 5) have mixed types. Specify dtype option on import or set low_memory=False.
  df_full = pd.read_csv('../references/Base_Carbone_V23.10.csv',


,Nom base français,Unité français,Total poste non décomposé,Statut de l'élément
10205,Gaz naturel,kgCO2e/kg,"3,22",Archivé
10206,Gaz naturel,kgCO2e/kg,"2,72",Archivé
10207,Gaz naturel,kgCO2e/kg,"0,498",Archivé
10208,Gaz naturel,kgCO2e/kg,"3,33",Archivé
10209,Gaz naturel,kgCO2e/kg,"2,81",Archivé
...,...,...,...,...
10639,"GNV, Gaz Naturel Comprimé pour véhicule routier",kgCO2e/GJ PCI,"56,6",Archivé
10640,"GNV, Gaz Naturel Comprimé pour véhicule routier",kgCO2e/GJ PCI,"11,4",Archivé
10641,"GNV, Gaz Naturel Comprrmé pour véhicule routier",kgCO2e/tonne,3465,Archivé
10642,"GNV, Gaz Naturel Comprrmé pour véhicule routier",kgCO2e/tonne,2809,Archivé


In [5]:
mask = (
    df_full['Nom base français'].str.contains('gaz naturel', case=False, na=False) &
    (df_full['Statut de l\'élément'] == 'Valide') &
    (df_full['Unité français'] == 'kgCO2e/kWh')
)
df_full[mask][['Nom base français', 'Unité français', 'Total poste non décomposé', 'Statut de l\'élément']]

,Nom base français,Unité français,Total poste non décomposé,Statut de l'élément


In [6]:
df_full['Statut de l\'élément'].unique()

<StringArray>
['Archivé', 'Valide générique', 'Valide spécifique']
Length: 3, dtype: str

In [7]:
mask = (
    df_full['Nom base français'].str.contains('gaz naturel', case=False, na=False) &
    (df_full['Statut de l\'élément'] == 'Valide générique') &
    (df_full['Unité français'] == 'kgCO2e/kWh')
)
df_full[mask][['Nom base français', 'Unité français', 'Total poste non décomposé', 'Statut de l\'élément']]

,Nom base français,Unité français,Total poste non décomposé,Statut de l'élément


In [8]:
mask = df_full['Nom base français'].str.contains('gaz naturel', case=False, na=False)
df_full[mask]['Unité français'].unique()

<StringArray>
[     'kgCO2e/kg', 'kgCO2e/kWh PCI',   'kgCO2e/litre', 'kgCO2e/tep PCI',
   'kgCO2e/tonne',  'kgCO2e/GJ PCI', 'kgCO2e/kWh PCS',  'kgCO2e/GJ PCS',
 'kgCO2e/tep PCS',  'kgCO2e/m3 (n)',     'GJ / tonne', 'kgCO2e/TEP PCI',
 'kgCO2e/TEP PCS',      'kgCO2e/m3',     'kgCO2e/kWh',        'kg / m³',
 'kgCO2e/KWh PCI',   'kgCO2e/Litre']
Length: 18, dtype: str

In [9]:
mask = (
    df_full['Nom base français'].str.contains('gaz naturel', case=False, na=False) &
    (df_full['Statut de l\'élément'] == 'Valide générique') &
    (df_full['Unité français'] == 'kgCO2e/kWh PCI')
)
df_full[mask][['Nom base français', 'Unité français', 'Total poste non décomposé', 'Statut de l\'élément']]

,Nom base français,Unité français,Total poste non décomposé,Statut de l'élément
10211,Gaz naturel,kgCO2e/kWh PCI,"0,243",Valide générique
10212,Gaz naturel,kgCO2e/kWh PCI,"0,205",Valide générique
10213,Gaz naturel,kgCO2e/kWh PCI,"0,038",Valide générique
10282,Gaz naturel - 2015,kgCO2e/kWh PCI,"0,227",Valide générique
10283,Gaz naturel - 2015,kgCO2e/kWh PCI,"0,187",Valide générique
10284,Gaz naturel - 2015,kgCO2e/kWh PCI,"0,0395",Valide générique
10303,Gaz naturel - 2022,kgCO2e/kWh PCI,"0,239",Valide générique
10304,Gaz naturel - 2022,kgCO2e/kWh PCI,"0,201",Valide générique
10305,Gaz naturel - 2022,kgCO2e/kWh PCI,"0,0382",Valide générique
10566,"GNC, Gaz Naturel Comprimé pour véhicule routier",kgCO2e/kWh PCI,"0,23",Valide générique


In [10]:
df_full[df_full.index.isin([10211, 10212, 10213])][['Nom base français', 'Nom attribut français', 'Unité français', 'Total poste non décomposé', 'CO2f']]

,Nom base français,Nom attribut français,Unité français,Total poste non décomposé,CO2f
10211,Gaz naturel,NaN,kgCO2e/kWh PCI,"0,243","0,227"
10212,Gaz naturel,NaN,kgCO2e/kWh PCI,"0,205","0,202"
10213,Gaz naturel,NaN,kgCO2e/kWh PCI,"0,038","0,0253"


In [11]:
df_full[df_full.index.isin([10211, 10212, 10213])][['Nom base français', 'Nom attribut français', 'Sous-localisation géographique français', 'Unité français', 'Total poste non décomposé', 'CO2f', 'CH4f', 'N2O']]

,Nom base français,Nom attribut français,Sous-localisation géographique français,Unité français,Total poste non décomposé,CO2f,CH4f,N2O
10211,Gaz naturel,NaN,NaN,kgCO2e/kWh PCI,"0,243","0,227","0,0132","2,46E-03"
10212,Gaz naturel,NaN,NaN,kgCO2e/kWh PCI,"0,205","0,202","5,02E-04","2,46E-03"
10213,Gaz naturel,NaN,NaN,kgCO2e/kWh PCI,"0,038","0,0253","0,0127",0


In [12]:
mask = (
    df_full['Nom base français'].str.contains('fioul', case=False, na=False) &
    (df_full['Statut de l\'élément'] == 'Valide générique') &
    (df_full['Unité français'] == 'kgCO2e/kWh PCI')
)
df_full[mask][['Nom base français', 'Nom attribut français', 'Unité français', 'Total poste non décomposé', 'CO2f']]

,Nom base français,Nom attribut français,Unité français,Total poste non décomposé,CO2f
9307,Fioul domestique,NaN,kgCO2e/kWh PCI,"0,324","0,317"
9308,Fioul domestique,NaN,kgCO2e/kWh PCI,"0,266","0,264"
9309,Fioul domestique,NaN,kgCO2e/kWh PCI,"0,0576","0,0527"
9310,Fioul domestique,NaN,kgCO2e/kWh PCI,"0,325","0,323"
9311,Fioul domestique,NaN,kgCO2e/kWh PCI,"0,272","0,27"
9312,Fioul domestique,NaN,kgCO2e/kWh PCI,"0,0527","0,0527"
9331,Fioul domestique,NaN,kgCO2e/kWh PCI,"0,308",NaN
9332,Fioul domestique,NaN,kgCO2e/kWh PCI,"0,271",NaN
9333,Fioul domestique,NaN,kgCO2e/kWh PCI,"0,037",NaN
9343,Fioul domestique,NaN,kgCO2e/kWh PCI,"0,3",NaN


In [13]:
df_full[df_full.index.isin([9307, 9308, 9309])][['Nom base français', 'Commentaire français', 'CO2f', 'CH4f', 'N2O']]

,Nom base français,Commentaire français,CO2f,CH4f,N2O
9307,Fioul domestique,NaN,"0,317","5,31E-03","1,47E-03"
9308,Fioul domestique,NaN,"0,264","4,32E-04","1,47E-03"
9309,Fioul domestique,NaN,"0,0527","4,88E-03",0


In [14]:
df_full[df_full.index.isin([9307, 9308, 9309])][['Nom base français', 'Nom attribut français', 'Type de l\'élément', 'Sous-localisation géographique français', 'CO2f', 'CH4f', 'N2O', 'Total poste non décomposé']]

,Nom base français,Nom attribut français,Type de l'élément,Sous-localisation géographique français,CO2f,CH4f,N2O,Total poste non décomposé
9307,Fioul domestique,NaN,Facteur d'émission,NaN,"0,317","5,31E-03","1,47E-03","0,324"
9308,Fioul domestique,NaN,Facteur d'émission,NaN,"0,264","4,32E-04","1,47E-03","0,266"
9309,Fioul domestique,NaN,Facteur d'émission,NaN,"0,0527","4,88E-03",0,"0,0576"


In [ ]:
[c for c in df_full.columns if 'poste' in c.lower() or 'local' in c.lower()]

In [ ]:
df_full[df_full.index.isin([9307, 9308, 9309])][['Nom base français', 'Localisation géographique', 'Sous-localisation géographique français', 'Type poste', 'Nom poste français', 'Total poste non décomposé']]

In [ ]:
df_full[df_full.index.isin([10211, 10212, 10213])][['Nom base français', 'Localisation géographique', 'Type poste', 'Total poste non décomposé', 'CO2f']]

In [ ]:
fuels = {
    'coal': 'houille',
    'lpg': 'gaz de pétrole liquéfié',
    'diesel': 'gazole'
}

for name, keyword in fuels.items():
    print(f"\n=== {name} ({keyword}) ===")
    mask = (
        df_full['Nom base français'].str.contains(keyword, case=False, na=False) &
        (df_full['Statut de l\'élément'] == 'Valide générique') &
        (df_full['Unité français'] == 'kgCO2e/kWh PCI')
    )
    print(df_full[mask][['Nom base français', 'Type poste', 'Total poste non décomposé', 'CO2f']].to_string())

In [ ]:
mask = (
    df_full['Nom base français'].str.contains('GPL|propane|butane', case=False, na=False) &
    (df_full['Statut de l\'élément'] == 'Valide générique') &
    (df_full['Unité français'] == 'kgCO2e/kWh PCI')
)
df_full[mask][['Nom base français', 'Type poste', 'Total poste non décomposé', 'CO2f']].to_string()

In [ ]:
mask = (
    df_full['Nom base français'].str.contains('gazole', case=False, na=False) &
    (df_full['Statut de l\'élément'] == 'Valide générique') &
    (df_full['Unité français'] == 'kgCO2e/kWh PCI')
)
df_full[mask][['Nom base français', 'Type poste', 'Localisation géographique', 'Total poste non décomposé', 'CO2f', 'Commentaire français']].to_string()

In [16]:
mask = (
    df_full['Nom base français'].str.contains('électricité', case=False, na=False) &
    (df_full['Statut de l\'élément'] == 'Valide générique') &
    (df_full['Localisation géographique'].str.contains('France', case=False, na=False))
)
df_full[mask][['Nom base français', 'Type poste', 'Localisation géographique', 
               'Unité français', 'Total poste non décomposé', 
               'Commentaire français']].to_string()

'     Nom base français  Type poste Localisation géographique Unité français Total poste non décomposé                                                                                                                                                                                                                                                                                     Commentaire français\n7667       Électricité         NaN       France continentale     kgCO2e/kWh                    0,0156                                                                                                                                                                                                                                                                                                      NaN\n7668       Électricité  Combustion       France continentale     kgCO2e/kWh                         0                                                                                                  

In [18]:
[c for c in df_full.columns if 'local' in c.lower()]

['Localisation géographique',
 'Sous-localisation géographique français',
 'Sous-localisation géographique anglais',
 'Sous-localisation géographique espagnol']

In [19]:
mask = (
    df_full['Nom base français'].str.contains('mix moyen|mix réseau', case=False, na=False) &
    (df_full['Statut de l\'élément'] == 'Valide générique') &
    (df_full['Localisation géographique'].str.contains('France', case=False, na=False)) &
    (df_full['Unité français'] == 'kgCO2e/kWh')
)
df_full[mask][['Nom base français', 'Type poste', 'Localisation géographique',
               'Unité français', 'Total poste non décomposé']].to_string()

'                 Nom base français                 Type poste Localisation géographique Unité français Total poste non décomposé\n8220  Electricité/2025 - mix moyen                        NaN       France continentale     kgCO2e/kWh                    0,0461\n8221  Electricité/2025 - mix moyen   Combustion à la centrale       France continentale     kgCO2e/kWh                    0,0291\n8222  Electricité/2025 - mix moyen                      Amont       France continentale     kgCO2e/kWh                    0,0127\n8223  Electricité/2025 - mix moyen  Transport et distribution       France continentale     kgCO2e/kWh                  4,30E-03'

In [20]:
mask = (
    df_full['Nom base français'].str.contains('mix moyen', case=False, na=False) &
    (df_full['Statut de l\'élément'] == 'Valide générique') &
    (df_full['Localisation géographique'].str.contains('France', case=False, na=False)) &
    (df_full['Unité français'] == 'kgCO2e/kWh')
)
df_full[mask][['Nom base français', 'Type poste', 'Localisation géographique',
               'Unité français', 'Total poste non décomposé']].to_string()

'                 Nom base français                 Type poste Localisation géographique Unité français Total poste non décomposé\n8220  Electricité/2025 - mix moyen                        NaN       France continentale     kgCO2e/kWh                    0,0461\n8221  Electricité/2025 - mix moyen   Combustion à la centrale       France continentale     kgCO2e/kWh                    0,0291\n8222  Electricité/2025 - mix moyen                      Amont       France continentale     kgCO2e/kWh                    0,0127\n8223  Electricité/2025 - mix moyen  Transport et distribution       France continentale     kgCO2e/kWh                  4,30E-03'

In [21]:
mask = (
    df_full['Nom base français'].str.contains('mix moyen|electricité', case=False, na=False) &
    (df_full['Statut de l\'élément'] == 'Valide générique') &
    (df_full['Localisation géographique'].str.contains('Allemagne', case=False, na=False)) &
    (df_full['Unité français'] == 'kgCO2e/kWh')
)
df_full[mask][['Nom base français', 'Type poste', 'Localisation géographique',
               'Unité français', 'Total poste non décomposé']].to_string()

'Empty DataFrame\nColumns: [Nom base français, Type poste, Localisation géographique, Unité français, Total poste non décomposé]\nIndex: []'

In [22]:
mask = (
    df_full['Nom base français'].str.contains('photovoltaïque|éolien|renouvelable', case=False, na=False) &
    (df_full['Statut de l\'élément'] == 'Valide générique') &
    (df_full['Localisation géographique'].str.contains('France', case=False, na=False)) &
    (df_full['Unité français'] == 'kgCO2e/kWh') &
    (df_full['Type poste'].isna())
)
df_full[mask][['Nom base français', 'Localisation géographique',
               'Unité français', 'Total poste non décomposé']].to_string()

'Empty DataFrame\nColumns: [Nom base français, Localisation géographique, Unité français, Total poste non décomposé]\nIndex: []'

In [23]:
mask = (
    df_full['Nom base français'].str.contains('renouvelable|ENR', case=False, na=False) &
    (df_full['Statut de l\'élément'] == 'Valide générique') &
    (df_full['Unité français'] == 'kgCO2e/kWh') &
    (df_full['Type poste'].isna())
)
df_full[mask][['Nom base français', 'Localisation géographique',
               'Unité français', 'Total poste non décomposé']].to_string()

'Empty DataFrame\nColumns: [Nom base français, Localisation géographique, Unité français, Total poste non décomposé]\nIndex: []'

In [24]:
mask = (
    df_full['Nom base français'].str.contains('nucléaire|nuclear', case=False, na=False) &
    (df_full['Statut de l\'élément'] == 'Valide générique') &
    (df_full['Unité français'] == 'kgCO2e/kWh') &
    (df_full['Type poste'].isna())
)
df_full[mask][['Nom base français', 'Localisation géographique',
               'Unité français', 'Total poste non décomposé']].to_string()

'Empty DataFrame\nColumns: [Nom base français, Localisation géographique, Unité français, Total poste non décomposé]\nIndex: []'